In [42]:
import joblib
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist

In [43]:
# Cargar modelo completo
modelo_clasificador = joblib.load('./binary/model_clasificador_lix.joblib')

scaler = modelo_clasificador['scaler']
pca_model = modelo_clasificador['pca']
col_names = modelo_clasificador['columnas_originales']
cluster_centers = modelo_clasificador['cluster_centers_pca']

In [44]:
def clasificar_muestra(chemical_res, scaler, pca_model, cluster_centers):
    """
    chemical_res: Una lista o array con los 16 valores químicos.
    """
    # 1. Asegurar que los datos tengan la forma correcta (1, 16)
    # muestra = np.array(chemical_res).reshape(1, -1)
    
    # 2. Escalado (Fundamental: usa la media y desviación del entrenamiento)
    muestra_escalada = scaler.transform(chemical_res)
    
    # 3. Reducción PCA (Bajar de 16 a las componentes que definiste)
    muestra_pca = pca_model.transform(muestra_escalada)
    
    # --- AJUSTE DE DIMENSIONES SI ES NECESARIO ---
    # Si tu ABC solo optimizó 10 componentes pero el PCA entrega 15 (como vimos en el error anterior)
    # Tomamos solo las primeras 10 componentes que son las que el ABC conoce
    muestra_pca_reducida = muestra_pca[:, :10] 
    
    # 4. Cálculo de distancias a los 3 centroides
    # 'centroides' tiene forma (3, 10)
    distancias = cdist(muestra_pca_reducida, cluster_centers, metric='euclidean')
    
    # 5. Obtener el cluster más cercano
    cluster_asignado = np.argmin(distancias)
    
    return cluster_asignado, distancias

In [ ]:
# Supongamos que tienes una nueva muestra de agua con 16 parámetros:
col_names = ['cadmio','cobre','coliformes_termotolerantes','coliformes_totales','cromo','dbo5',
             'dqo','hierro','manganeso','niquel','ph','plomo','solidos_suspendidos',
             'solidos_totales','zinc']

muestras_quimicas = np.array([
    [0.5, 2.89, 13000.0, 13000.0, 2.0, 0.35, 7.75, 1.093, 0.03675, 5.2, 8.44, 10.0, 0.0,293.0, 0.1172],
    [0.5,10.36,2800.0,8000.0,19.7,0.4,3.7,0.262,0.0706,50.0,7.723999999999999,10.0,121.0,434.0,0.032],
    [20.0,13.4,16000.0,16000.0,50.0,0.3,12.0,2.106,0.035,27.7,8.1,100.0,37.0,455.0,0.0239],
    [10.52,143.40599999999998,5400.0,9200.0,416.03000000000003,117.22999999999999,7.725,20.644,1.458,373.83000000000004,8.32,64.2,120.0,12.344,1.252],
])


for i in range(muestras_quimicas.shape[0]):
    muestra_quimica = pd.DataFrame([muestras_quimicas[i]], columns=col_names)
    display(muestra_quimica)

    cluster, dits = clasificar_muestra(
        chemical_res=muestra_quimica, 
        scaler=scaler, 
        pca_model=pca_model, 
        cluster_centers=cluster_centers
    )

    print(f"La muestra ha sido clasificada en el Cluster: {cluster}, distancias: {dits}")

[[5.00000e-01 2.89000e+00 1.30000e+04 1.30000e+04 2.00000e+00 3.50000e-01
  7.75000e+00 1.09300e+00 3.67500e-02 5.20000e+00 8.44000e+00 1.00000e+01
  0.00000e+00 2.93000e+02 1.17200e-01]
 [5.00000e-01 1.03600e+01 2.80000e+03 8.00000e+03 1.97000e+01 4.00000e-01
  3.70000e+00 2.62000e-01 7.06000e-02 5.00000e+01 7.72400e+00 1.00000e+01
  1.21000e+02 4.34000e+02 3.20000e-02]
 [2.00000e+01 1.34000e+01 1.60000e+04 1.60000e+04 5.00000e+01 3.00000e-01
  1.20000e+01 2.10600e+00 3.50000e-02 2.77000e+01 8.10000e+00 1.00000e+02
  3.70000e+01 4.55000e+02 2.39000e-02]
 [1.05200e+01 1.43406e+02 5.40000e+03 9.20000e+03 4.16030e+02 1.17230e+02
  7.72500e+00 2.06440e+01 1.45800e+00 3.73830e+02 8.32000e+00 6.42000e+01
  1.20000e+02 1.23440e+01 1.25200e+00]]


,cadmio,cobre,coliformes_termotolerantes,coliformes_totales,cromo,dbo5,dqo,hierro,manganeso,niquel,ph,plomo,solidos_suspendidos,solidos_totales,zinc
0,0.5,2.89,13000.0,13000.0,2.0,0.35,7.75,1.093,0.03675,5.2,8.44,10.0,0.0,293.0,0.1172


La muestra ha sido clasificada en el Cluster: 1, distancias: [[20.96195404  1.55143093  4.40056662]]


,cadmio,cobre,coliformes_termotolerantes,coliformes_totales,cromo,dbo5,dqo,hierro,manganeso,niquel,ph,plomo,solidos_suspendidos,solidos_totales,zinc
0,0.5,10.36,2800.0,8000.0,19.7,0.4,3.7,0.262,0.0706,50.0,7.724,10.0,121.0,434.0,0.032


La muestra ha sido clasificada en el Cluster: 1, distancias: [[20.83432569  1.26975451  3.90922314]]


,cadmio,cobre,coliformes_termotolerantes,coliformes_totales,cromo,dbo5,dqo,hierro,manganeso,niquel,ph,plomo,solidos_suspendidos,solidos_totales,zinc
0,20.0,13.4,16000.0,16000.0,50.0,0.3,12.0,2.106,0.035,27.7,8.1,100.0,37.0,455.0,0.0239


La muestra ha sido clasificada en el Cluster: 1, distancias: [[20.82971073  1.15634626  3.6936538 ]]


,cadmio,cobre,coliformes_termotolerantes,coliformes_totales,cromo,dbo5,dqo,hierro,manganeso,niquel,ph,plomo,solidos_suspendidos,solidos_totales,zinc
0,10.52,143.406,5400.0,9200.0,416.03,117.23,7.725,20.644,1.458,373.83,8.32,64.2,120.0,12.344,1.252


La muestra ha sido clasificada en el Cluster: 2, distancias: [[20.93740781  3.0034923   1.96877792]]
